# Movio — Indic Voice Streaming Server (Kaggle GPU)

This notebook clones the Movio repo, installs dependencies, loads the TTS model on the Kaggle GPU, and exposes the server + frontend via a Cloudflare tunnel (free, no account needed).

**Steps:**
1. Clone repo & install deps
2. Authenticate HuggingFace (for model download)
3. Load model
4. Start server + Cloudflare tunnel

**After running all cells**, click the printed `https://...trycloudflare.com` link to open the frontend. It auto-connects to the backend — no URL pasting needed.

## 1. Environment Check

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 2. Clone Repo & Install Dependencies

**After this cell finishes, restart the kernel** (Run → Restart Session) then run all cells from the top again. This is needed because Kaggle's base image caches an older `transformers` version.

In [ ]:
!rm -rf /kaggle/working/movio-2
!git clone https://github.com/tripathiji1312/movio_new.git /kaggle/working/movio-2
!pip install -q -r /kaggle/working/movio-2/requirements.txt
!pip install -q nest_asyncio
print("\nInstallation complete. Restart the kernel now, then run all cells again.")

## 3. HuggingFace Login

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("HuggingFace login successful!")

## 4. Load Model

This loads the Indic Parler-TTS model onto the GPU and runs a warm-up generation.

In [ ]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import sys
sys.path.insert(0, "/kaggle/working/movio-2")

from backend.tts_engine import engine
engine.load()

## 5. Start Server + Cloudflare Tunnel

This starts the FastAPI server (serves both API + frontend) and creates a free Cloudflare tunnel.

Click the printed URL to open the app — it auto-connects, no pasting needed.

In [ ]:
import nest_asyncio
import uvicorn
import threading
import subprocess
import re
import time

from backend.server import app

nest_asyncio.apply()

PORT = 8000

# Start FastAPI server
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

# Download and start cloudflared (free, no account/token needed)
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared

cf_process = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for the tunnel URL to appear in cloudflared's stderr
tunnel_url = None
for _ in range(30):
    line = cf_process.stderr.readline().decode("utf-8", errors="replace")
    match = re.search(r"(https://[\w-]+\.trycloudflare\.com)", line)
    if match:
        tunnel_url = match.group(1)
        break

if tunnel_url:
    print("=" * 60)
    print("SERVER IS LIVE!")
    print("=" * 60)
    print(f"\nOpen this link in your browser:\n")
    print(f"  {tunnel_url}")
    print(f"\nThe frontend auto-connects — no URL pasting needed.")
    print("=" * 60)
else:
    print("ERROR: Could not get Cloudflare tunnel URL.")
    print("Stderr so far:")
    print(cf_process.stderr.read().decode("utf-8", errors="replace")[:2000])

## 6. (Optional) Quick Test

Run this cell to do a quick local synthesis test to verify everything works before using the frontend.

In [ ]:
import numpy as np
from IPython.display import Audio, display
from backend.normalizer import normalize_text
from backend.chunker import split_into_sentences
from backend.config import VOICE_DESCRIPTION

test_text = "Your OTP is {{OTP:483927}}. Please do not share this with anyone."
chunks = [normalize_text(c) for c in split_into_sentences(test_text)]
print(f"Chunks: {chunks}")

for chunk in chunks:
    audio_pieces = []
    for piece, t in engine.stream_generate(chunk, VOICE_DESCRIPTION):
        audio_pieces.append(piece)
    audio = np.concatenate(audio_pieces)
    display(Audio(audio, rate=engine.sample_rate, autoplay=False))
    print(f"Generated {len(audio)/engine.sample_rate:.2f}s of audio")